# Machine Learning Agent with LangGraph

A multi-agent ML pipeline that demonstrates **reasoning and planning** using LangGraph + Claude.

```
┌─────────────────────────────────────────────────────────────────┐
│                      SUPERVISOR AGENT                           │
│   Plans strategy · Evaluates results · Routes between agents   │
└────────┬──────────────────────────────────────────────┬────────┘
         │                                              │
         ▼                                              ▼
  ┌─────────────┐   ┌─────────────┐   ┌──────────────┐   ┌───────────┐
  │    Data     │──▶│    Data     │──▶│    Model     │──▶│  Testing  │
  │ Exploration │   │    Prep     │   │   Training   │   │  Agent    │
  └─────────────┘   └──────┬──────┘   └──────────────┘   └───────────┘
                           │  ▲  Feedback loop if accuracy < threshold
                           └──┘
```

**Flow:**
1. Supervisor plans and routes to **Data Exploration**
2. Exploration findings inform routing to **Data Prep**
3. Data Prep chains directly to **Model Training**
4. Training results return to **Supervisor** for evaluation
5. If accuracy < threshold → Supervisor routes back to **Data Prep** (feedback loop)
6. If accuracy ≥ threshold → Supervisor routes to **Testing**
7. Testing completes → Supervisor ends the pipeline

Each step is logged with full LLM reasoning visible.

In [53]:
# ── Architecture diagram (opens the HTML visual in an IFrame) ─────────────────
from IPython.display import IFrame
IFrame(src='html/ml_pipeline_architecture.html', width='100%', height='860px')

In [41]:
%pip install -q langgraph langchain-openai scikit-learn pandas numpy

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [54]:
import os
import getpass
import warnings
warnings.filterwarnings('ignore')

from typing import TypedDict, Literal
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

from IPython.display import display, HTML

print("✓ Imports complete")

✓ Imports complete


In [55]:
# ── Shared pipeline state ──────────────────────────────────────────────────────
class MLPipelineState(TypedDict):
    task_description: str
    dataset_info: dict          # filled by data_exploration_agent
    prep_info: dict             # filled by data_prep_agent
    model_results: dict         # filled by model_training_agent
    test_results: dict          # filled by testing_agent
    supervisor_memo: str        # accumulating supervisor scratchpad
    reasoning_log: list         # every agent's reasoning, in order
    iteration_count: int        # how many train/prep cycles we've run
    max_iterations: int
    accuracy_threshold: float
    next_agent: str             # supervisor's routing decision
    next_agent_instruction: str # supervisor's instruction to the next agent
    pipeline_complete: bool

# ── Global data store (persists ML artifacts between agent calls) ──────────────
# LangGraph state is JSON-serialisable; numpy arrays / sklearn objects live here.
data_store: dict = {}

# ── Display utilities ──────────────────────────────────────────────────────────
AGENT_COLORS = {
    "SUPERVISOR":        "#9C6FD6",
    "DATA EXPLORATION":  "#4C9FE8",
    "DATA PREP":         "#F5A623",
    "MODEL TRAINING":    "#5CB85C",
    "TESTING":           "#E8604C",
}

def log_agent(agent_name: str, content: str) -> None:
    color = AGENT_COLORS.get(agent_name, "#888888")
    ts = datetime.now().strftime("%H:%M:%S")
    display(HTML(f"""
    <div style="margin:8px 0;padding:12px 14px;border-left:4px solid {color};
                background:transparent;border-radius:4px;font-family:monospace;font-size:13px;">
      <b style="color:{color};">[{ts}] {agent_name}</b>
      <pre style="margin:6px 0 0;white-space:pre-wrap;color:inherit;">{content}</pre>
    </div>"""))

print("✓ State and utilities defined")

✓ State and utilities defined


In [56]:
# ── API key (entered securely — not stored in notebook) ───────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# ── LLM and structured output for supervisor ───────────────────────────────────
llm = ChatOpenAI(model="gpt-4o", temperature=0)

class SupervisorDecision(BaseModel):
    """Supervisor's structured routing decision."""
    reasoning: str = Field(
        description="Detailed reasoning about the current pipeline state and what has been accomplished so far")
    plan: str = Field(
        description="What you plan to do next and the specific reason why")
    next_agent: Literal["data_exploration", "data_prep", "testing", "end"] = Field(
        description="The next agent to activate. Note: data_prep automatically chains to model_training.")
    instruction: str = Field(
        description="Specific actionable instruction for the next agent")

supervisor_llm = llm.with_structured_output(SupervisorDecision)

print("✓ LLM configured — model: gpt-4o")

✓ LLM configured — model: gpt-4o


## Agent Definitions

Each agent:
1. Receives the shared pipeline state
2. Runs its ML code (real sklearn operations)
3. Calls the LLM to reason about findings
4. Returns a state update dict (LangGraph merges it)

In [57]:
# ── Data Exploration Agent ─────────────────────────────────────────────────────
def data_exploration_agent(state: MLPipelineState) -> dict:
    log_agent("DATA EXPLORATION", "Loading breast cancer dataset and profiling...")

    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df["target"] = data.target

    data_store.update({
        "raw_data": data,
        "df": df,
        "feature_names": list(data.feature_names),
        "target_names": list(data.target_names),
    })

    feat_df = df.drop("target", axis=1)
    class_dist = {str(data.target_names[k]): int(v)
                  for k, v in df["target"].value_counts().items()}

    dataset_info = {
        "n_samples":         int(len(df)),
        "n_features":        int(len(data.feature_names)),
        "feature_names":     list(data.feature_names),
        "target_names":      list(data.target_names),
        "class_distribution": class_dist,
        "missing_values":    int(df.isnull().sum().sum()),
        "feature_mean_range": [round(float(feat_df.mean().min()), 4),
                               round(float(feat_df.mean().max()), 4)],
        "feature_std_range":  [round(float(feat_df.std().min()), 4),
                               round(float(feat_df.std().max()), 4)],
    }

    prompt = f"""Dataset: Breast Cancer Wisconsin Diagnostic
Samples: {dataset_info['n_samples']} | Features: {dataset_info['n_features']} (all numerical) | Missing: {dataset_info['missing_values']}
Classes: {class_dist}  (class balance check: {'balanced' if min(class_dist.values())/max(class_dist.values()) > 0.6 else 'imbalanced'})
Feature value range is large: means span {dataset_info['feature_mean_range']}, stds span {dataset_info['feature_std_range']}

Supervisor instruction: {state.get('next_agent_instruction', 'Perform thorough EDA')}

Provide:
1. Key findings (data quality, class balance, feature scale)
2. Risks that could hurt model performance
3. Preprocessing recommendations
4. Suggested model families to try"""

    response = llm.invoke([
        SystemMessage(content="You are a data exploration agent for ML pipelines. Be concise and actionable."),
        HumanMessage(content=prompt),
    ])

    log_agent("DATA EXPLORATION", response.content)

    return {
        "dataset_info": dataset_info,
        "reasoning_log": state["reasoning_log"] + [
            f"[DATA EXPLORATION]\n{response.content}"
        ],
        "supervisor_memo": state["supervisor_memo"] + f"\n[Exploration] {response.content[:250]}...",
    }

In [58]:
# ── Data Prep Agent ────────────────────────────────────────────────────────────
# Iteration 0 : limited features, no scaling  (deliberately weak — triggers feedback loop)
# Iteration 1+: all features + StandardScaler (improved after supervisor feedback)
def data_prep_agent(state: MLPipelineState) -> dict:
    iteration = state["iteration_count"]
    log_agent("DATA PREP", f"Preparing data — pass {iteration + 1}...")

    df = data_store["df"]
    X, y = df.drop("target", axis=1), df["target"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    if iteration == 0:
        # Baseline: first 10 features only, no scaling
        selected = list(X.columns[:10])
        X_tr = X_train[selected].values
        X_te = X_test[selected].values
        description = "10 features (first 10 columns), no scaling — baseline pass"
    else:
        # Improved: all 30 features + StandardScaler
        selected = list(X.columns)
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
        data_store["scaler"] = scaler
        description = f"All {len(selected)} features + StandardScaler — improved after accuracy feedback"

    data_store.update({
        "X_train": X_tr, "X_test": X_te,
        "y_train": y_train.values, "y_test": y_test.values,
        "selected_features": selected,
    })

    prep_info = {
        "description":    description,
        "n_features_used": len(selected),
        "train_size":      int(len(X_tr)),
        "test_size":       int(len(X_te)),
        "scaling":         iteration > 0,
    }

    prev_acc = state.get("model_results", {}).get("accuracy")
    context = (
        f"\nPrevious model accuracy was {prev_acc:.4f} — below threshold {state['accuracy_threshold']}. "
        "Supervisor instructed: improve preprocessing."
        if prev_acc else ""
    )

    response = llm.invoke([
        SystemMessage(content="You are a data preparation agent for ML pipelines."),
        HumanMessage(content=(
            f"Completed: {description}\n"
            f"Train: {prep_info['train_size']} samples | Test: {prep_info['test_size']} samples"
            f"{context}\n\n"
            "Explain what you did and why it should improve model performance."
        )),
    ])

    log_agent("DATA PREP", f"{description}\n\n{response.content}")

    return {
        "prep_info": prep_info,
        "reasoning_log": state["reasoning_log"] + [
            f"[DATA PREP — pass {iteration+1}]\n{description}\n{response.content}"
        ],
    }

In [59]:
# ── Model Training Agent ───────────────────────────────────────────────────────
# Iteration 0 : Logistic Regression on limited unscaled features (~0.90–0.93)
# Iteration 1+: Random Forest on all scaled features (~0.97+)
def model_training_agent(state: MLPipelineState) -> dict:
    iteration = state["iteration_count"]
    log_agent("MODEL TRAINING", f"Training — pass {iteration + 1}...")

    X_train, y_train = data_store["X_train"], data_store["y_train"]
    X_test,  y_test  = data_store["X_test"],  data_store["y_test"]

    if iteration == 0:
        model = LogisticRegression(max_iter=200, random_state=42)
        model_name = "Logistic Regression (max_iter=200, 10 unscaled features)"
    else:
        model = RandomForestClassifier(n_estimators=150, random_state=42)
        model_name = "Random Forest (n_estimators=150, all features scaled)"

    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    accuracy = float(accuracy_score(y_test, y_pred))
    report   = classification_report(y_test, y_pred,
                                     target_names=data_store["target_names"])

    data_store.update({"model": model, "y_pred": y_pred})

    threshold = state["accuracy_threshold"]
    status = "✓ PASS" if accuracy >= threshold else f"✗ FAIL  (threshold {threshold})"
    log_agent("MODEL TRAINING",
              f"Model : {model_name}\nAccuracy: {accuracy:.4f}  {status}\n\n{report}")

    response = llm.invoke([
        SystemMessage(content="You are a model training agent. Analyse results and recommend next steps."),
        HumanMessage(content=(
            f"Model: {model_name}\nAccuracy: {accuracy:.4f}  ({status})\n"
            f"Threshold: {threshold}\n\n{report}\n\n"
            "Analyse the classification report. If below threshold, suggest what the "
            "data prep agent should change next iteration."
        )),
    ])

    log_agent("MODEL TRAINING", f"Analysis:\n{response.content}")

    return {
        "model_results": {
            "model_name":        model_name,
            "accuracy":          accuracy,
            "classification_report": report,
            "iteration":         iteration + 1,
            "passed_threshold":  accuracy >= threshold,
        },
        "reasoning_log": state["reasoning_log"] + [
            f"[MODEL TRAINING — pass {iteration+1}]\n"
            f"Model: {model_name}\nAccuracy: {accuracy:.4f}  {status}\n{response.content}"
        ],
    }

In [60]:
# ── Testing / Evaluation Agent ─────────────────────────────────────────────────
def testing_agent(state: MLPipelineState) -> dict:
    log_agent("TESTING", "Running final comprehensive evaluation...")

    y_test = data_store["y_test"]
    y_pred = data_store["y_pred"]
    model  = data_store["model"]

    accuracy = float(accuracy_score(y_test, y_pred))
    report   = classification_report(y_test, y_pred, target_names=data_store["target_names"])
    cm       = confusion_matrix(y_test, y_pred).tolist()

    top_features = None
    if hasattr(model, "feature_importances_"):
        names = data_store["selected_features"]
        pairs = sorted(zip(names, model.feature_importances_),
                       key=lambda x: x[1], reverse=True)[:5]
        top_features = [(n, round(float(v), 4)) for n, v in pairs]

    log_agent("TESTING", f"Final accuracy: {accuracy:.4f}\n\n{report}")
    if top_features:
        fi_str = "\n".join(f"  {n}: {v:.4f}" for n, v in top_features)
        log_agent("TESTING", f"Top-5 feature importances:\n{fi_str}")

    response = llm.invoke([
        SystemMessage(content="You are a testing and evaluation agent. Write a concise final report."),
        HumanMessage(content=(
            f"Final evaluation results:\n"
            f"Model: {state['model_results']['model_name']}\n"
            f"Final accuracy: {accuracy:.4f}\n"
            f"Pipeline iterations needed: {state['iteration_count'] + 1}\n"
            f"Confusion matrix: {cm}\n"
            f"Top features: {top_features}\n\n"
            f"{report}\n\n"
            "Write a final report covering: performance summary, key pipeline insights, "
            "production readiness, and suggested improvements."
        )),
    ])

    log_agent("TESTING", f"Final Report:\n{response.content}")

    return {
        "test_results": {
            "final_accuracy":         accuracy,
            "classification_report":  report,
            "confusion_matrix":       cm,
            "top_features":           top_features,
        },
        "pipeline_complete": True,
        "reasoning_log": state["reasoning_log"] + [
            f"[TESTING — FINAL]\nAccuracy: {accuracy:.4f}\n{response.content}"
        ],
    }

In [61]:
# ── Supervisor Agent ───────────────────────────────────────────────────────────
SUPERVISOR_SYSTEM = """You are the supervisor of an ML pipeline. Your job is to plan, 
evaluate, and orchestrate specialist sub-agents to build the best possible model.

Available routing options:
  data_exploration  — Profile and understand the dataset
  data_prep         — Prepare/engineer features (automatically chains to model_training)
  testing           — Final comprehensive evaluation (only after acceptable accuracy)
  end               — Close the pipeline (only after testing is complete)

Routing logic (follow strictly):
  1. dataset_info empty           → data_exploration
  2. prep_info empty              → data_prep
  3. model passed threshold       → testing
  4. model failed AND iter < max  → data_prep  (with specific improvement instruction)
  5. model failed AND iter ≥ max  → testing    (best effort)
  6. testing complete             → end

Be explicit in your reasoning. Your memo is cumulative — build on previous entries."""


def supervisor_agent(state: MLPipelineState) -> dict:
    log_agent("SUPERVISOR", "Evaluating pipeline state...")

    # Build a concise context string
    lines = [
        f"Task: {state['task_description']}",
        f"Accuracy threshold: {state['accuracy_threshold']}   "
        f"Iteration: {state['iteration_count']} / {state['max_iterations']}",
    ]

    if state.get("dataset_info"):
        di = state["dataset_info"]
        lines.append(
            f"Dataset: {di['n_samples']} samples, {di['n_features']} features, "
            f"classes={di['target_names']}, class_dist={di['class_distribution']}, "
            f"missing={di['missing_values']}, scale_spread=large")
    else:
        lines.append("Dataset: NOT YET EXPLORED")

    if state.get("prep_info"):
        lines.append(f"Data prep: {state['prep_info']['description']}")
    else:
        lines.append("Data prep: NOT YET DONE")

    if state.get("model_results"):
        mr = state["model_results"]
        status = "PASS" if mr["passed_threshold"] else "FAIL"
        lines.append(
            f"Training: {mr['model_name']} | Accuracy: {mr['accuracy']:.4f} [{status}]")
    else:
        lines.append("Training: NOT YET DONE")

    if state.get("test_results"):
        lines.append(f"Testing: COMPLETE — final accuracy {state['test_results']['final_accuracy']:.4f}")
    else:
        lines.append("Testing: NOT YET DONE")

    lines.append(f"\nSupervisor memo so far:\n{state['supervisor_memo'] or '(none)'}")

    decision = supervisor_llm.invoke([
        SystemMessage(content=SUPERVISOR_SYSTEM),
        HumanMessage(content="\n".join(lines)),
    ])

    log_agent("SUPERVISOR", (
        f"REASONING:\n{decision.reasoning}\n\n"
        f"PLAN:\n{decision.plan}\n\n"
        f"ROUTING → {decision.next_agent.upper()}\n"
        f"INSTRUCTION: {decision.instruction}"
    ))

    # Increment iteration when supervisor decides to retry data prep after a failure
    new_iteration = state["iteration_count"]
    if decision.next_agent == "data_prep" and state.get("model_results"):
        new_iteration += 1

    return {
        "next_agent":             decision.next_agent,
        "next_agent_instruction": decision.instruction,
        "iteration_count":        new_iteration,
        "supervisor_memo":        (
            state["supervisor_memo"]
            + f"\n\n[Iter {state['iteration_count']}] {decision.reasoning}"
        ),
        "reasoning_log": state["reasoning_log"] + [
            f"[SUPERVISOR]\n"
            f"REASONING: {decision.reasoning}\n\n"
            f"PLAN: {decision.plan}\n\n"
            f"ROUTING → {decision.next_agent.upper()}"
        ],
    }

In [62]:
# ── Graph construction ─────────────────────────────────────────────────────────
def route_supervisor(state: MLPipelineState) -> str:
    return state["next_agent"]


def build_pipeline() -> StateGraph:
    g = StateGraph(MLPipelineState)

    # Register nodes
    g.add_node("supervisor",       supervisor_agent)
    g.add_node("data_exploration", data_exploration_agent)
    g.add_node("data_prep",        data_prep_agent)
    g.add_node("model_training",   model_training_agent)
    g.add_node("testing",          testing_agent)

    # Entry point
    g.set_entry_point("supervisor")

    # Supervisor routes conditionally
    g.add_conditional_edges(
        "supervisor",
        route_supervisor,
        {
            "data_exploration": "data_exploration",
            "data_prep":        "data_prep",
            "testing":          "testing",
            "end":              END,
        },
    )

    # Fixed edges
    g.add_edge("data_exploration", "supervisor")   # explore → supervisor decides next
    g.add_edge("data_prep",        "model_training")  # prep always chains to training
    g.add_edge("model_training",   "supervisor")   # training results → supervisor evaluates
    g.add_edge("testing",          "supervisor")   # testing → supervisor issues end

    return g.compile()


pipeline = build_pipeline()
print("✓ Pipeline graph compiled")
print()
print("Graph edges:")
print("  supervisor ──(conditional)──▶ data_exploration | data_prep | testing | END")
print("  data_exploration ────────────▶ supervisor")
print("  data_prep ───────────────────▶ model_training")
print("  model_training ──────────────▶ supervisor")
print("  testing ─────────────────────▶ supervisor")

✓ Pipeline graph compiled

Graph edges:
  supervisor ──(conditional)──▶ data_exploration | data_prep | testing | END
  data_exploration ────────────▶ supervisor
  data_prep ───────────────────▶ model_training
  model_training ──────────────▶ supervisor
  testing ─────────────────────▶ supervisor


In [63]:
# ── LangGraph visual ──────────────────────────────────────────────────────────
# Renders the compiled graph as a Mermaid diagram via CDN (no extra installs needed).
from IPython.display import display, HTML

mermaid_code = pipeline.get_graph().draw_mermaid()

display(HTML(f"""
<div style="background:transparent;padding:10px;border-radius:10px;
            border:1px solid rgba(255,255,255,0.1);">
  <p style="font-family:monospace;font-size:11px;opacity:0.6;margin:0 0 10px;">
    LangGraph compiled pipeline — nodes and edges
  </p>
  <div class="mermaid">
{mermaid_code}
  </div>
</div>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
  mermaid.initialize({{ startOnLoad: true, theme: 'dark',
    themeVariables: {{ darkMode: true, background: 'transparent',
      primaryColor: '#1e1e3a', primaryTextColor: '#e8edff',
      primaryBorderColor: '#00f0ff', lineColor: '#a855f7',
      secondaryColor: '#12122e', tertiaryColor: '#0a0a24' }} }});
</script>
"""))

## Run the Pipeline

The pipeline will:
1. Supervisor plans → routes to Data Exploration
2. Data Exploration profiles the dataset → back to Supervisor
3. Supervisor routes to Data Prep (baseline: 10 features, no scaling)
4. Data Prep → Model Training (Logistic Regression) → ~0.90–0.93 accuracy → **FAIL**
5. Supervisor sees failure → routes back to Data Prep with improved instructions
6. Data Prep (all features + StandardScaler) → Model Training (Random Forest) → ~0.97+ → **PASS**
7. Supervisor routes to Testing for final evaluation
8. Testing complete → Supervisor ends the pipeline

In [51]:
display(HTML("""
<div style="border:2px solid #7986CB;border-radius:10px;
            padding:20px 24px;font-family:sans-serif;background:transparent;">
  <h2 style="margin:0 0 6px;color:inherit;">ML Agent Pipeline — Starting</h2>
  <p style="margin:0;font-size:13px;opacity:0.7;">
    Dataset: Breast Cancer Wisconsin · Target accuracy: ≥ 0.96 · Max iterations: 3 · Model: gpt-4o
  </p>
</div>
"""))
# Reset global data store for a clean run
data_store.clear()

initial_state: MLPipelineState = {
    "task_description": (
        "Build a binary classifier for breast cancer diagnosis "
        "(malignant vs benign). Target: accuracy ≥ 0.96."
    ),
    "dataset_info":           {},
    "prep_info":              {},
    "model_results":          {},
    "test_results":           {},
    "supervisor_memo":        "",
    "reasoning_log":          [],
    "iteration_count":        0,
    "max_iterations":         3,
    "accuracy_threshold":     0.96,
    "next_agent":             "",
    "next_agent_instruction": "",
    "pipeline_complete":      False,
}

final_state = pipeline.invoke(initial_state, {"recursion_limit": 30})

In [52]:
# ── Full Reasoning & Planning Log ──────────────────────────────────────────────
display(HTML("<hr/><h2 style='font-family:sans-serif;color:inherit;'>Full Reasoning & Planning Log</h2>"
             "<p style='font-family:sans-serif;color:inherit;opacity:0.7;'>Click any step to expand the agent's full reasoning.</p>"))

for i, entry in enumerate(final_state["reasoning_log"], 1):
    first_line = entry.split("\n")[0]
    color = "#888888"
    for name, c in AGENT_COLORS.items():
        if name in first_line.upper():
            color = c
            break
    display(HTML(f"""
    <details style="margin:5px 0;border-left:3px solid {color};background:transparent;
                    border-radius:3px;">
      <summary style="cursor:pointer;padding:8px 14px;font-family:monospace;
                      font-weight:bold;color:{color};font-size:13px;">
        Step {i} &nbsp;·&nbsp; {first_line}
      </summary>
      <pre style="padding:10px 16px;font-size:12px;white-space:pre-wrap;
                  color:inherit;margin:0;">{entry}</pre>
    </details>"""))

# ── Final summary card ─────────────────────────────────────────────────────────
if final_state.get("test_results"):
    tr = final_state["test_results"]
    mr = final_state["model_results"]
    iters = final_state["iteration_count"] + 1
    loop_note = (
        f"{iters} iterations — feedback loop triggered {iters - 1}x"
        if iters > 1 else "1 iteration — passed on first attempt"
    )
    tf_html = ""
    if tr.get("top_features"):
        tf_html = "<br>".join(f"  {n}: {v}" for n, v in tr["top_features"])
        tf_html = f"<p style='margin:4px 0;color:inherit;'><b>Top features:</b><br><code>{tf_html}</code></p>"

    display(HTML(f"""
    <div style="margin-top:20px;padding:18px 20px;background:transparent;
                border:2px solid #5CB85C;border-radius:8px;font-family:monospace;color:inherit;">
      <h3 style="margin:0 0 10px;color:#5CB85C;font-family:sans-serif;">Pipeline Complete</h3>
      <p style="margin:4px 0;color:inherit;"><b>Final model:</b> {mr['model_name']}</p>
      <p style="margin:4px 0;color:inherit;"><b>Final accuracy:</b> {tr['final_accuracy']:.4f}</p>
      <p style="margin:4px 0;color:inherit;"><b>Iterations:</b> {loop_note}</p>
      <p style="margin:4px 0;color:inherit;"><b>Confusion matrix:</b> {tr['confusion_matrix']}</p>
      {tf_html}
    </div>"""))